In [1]:
import sys
import os
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(".."))
from churn.modelo import dividir_datos, construir_modelo
from churn.datos import cargar_datos
from churn.eda import graficar_tasa
from churn.metricas import precision_at_k, lift_at_k, tabla_ganancias
from churn.explicacion import calcular_shap
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

/home/diego/Proyectos/churn-sdg/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [3]:
df = cargar_datos("../data/dataset.csv")

In [4]:
X_train, X_test, y_train, y_test = dividir_datos(df)

In [5]:
num_cols = [
    "rev_Mean",         # Mean monthly revenue (charge amount)
    "mou_Mean",         # Mean number of monthly minutes of use
    "ovrrev_Mean",      # Mean overage revenue
    "comp_dat_Mean",    # Mean number of completed data calls
    "custcare_Mean",    # Mean number of customer care calls
    "phones",           # Number of handsets issued
    "months",           # Total number of months in service
    "eqpdays",          # Number of days (age) of current mobile phone
    "change_mou",       # Percentage change in monthly minutes of use vs previous three month average
    "change_rev",       # Percentage change in monthly revenue vs previous three month average
    "drop_vce_Mean",    # Mean number of dropped (failed) voice calls
    "drop_dat_Mean",    # Mean number of dropped (failed) data calls
    "drop_blk_Mean",    # Mean number of dropped or blocked calls
    "blck_vce_Mean",    # Mean number of blocked (failed) voice calls
    "blck_dat_Mean",    # Mean number of blocked (failed) data calls
    "unan_dat_Mean",    # Mean number of unanswered data calls
    "totmrc_Mean",      # Mean total monthly recurring charge
    "uniqsubs",         # Number of unique subscribers in the household
    "roam_Mean",        # Mean number of roaming calls
    "datovr_Mean",      # Mean revenue of data overage
    "da_Mean",          # Mean number of directory assisted calls
    "recv_sms_Mean",    # Mean number of received sms
    "callfwdv_Mean",    # Mean number of call forwarding calls
    "callwait_Mean",    # Mean number of call waiting calls
    "threeway_Mean",    # Mean number of three way calls
    "hnd_price",        # Current handset price
]

In [6]:
cat_cols = [
    "refurb_new",   # Handset: refurbished or new
    "infobase",     # InfoBase match
    "dualband",     # Dualband
    "asl_flag",     # Account spending limit
    "hnd_webcap",   # Handset web capability
]

In [7]:
config_regularizada = LGBMClassifier(
    learning_rate= 0.05,   
    n_estimators= 300,      
    num_leaves= 15,        
    max_depth= 5,          
    min_child_samples= 50, 
    colsample_bytree= 0.8, 
    reg_alpha= 0.1,        
    reg_lambda= 0.5,       
    random_state= 42,
    verbose=-1
)

In [8]:
pipeline_lgbm = construir_modelo(num_cols, cat_cols,config_regularizada,False)
pipeline_lgbm.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('limpieza', ...), ('pre', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](98,)","['rev_Mean','mou_Mean','totmrc_Mean',...,'kid16_17','creditcd','eqpdays']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,98
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function lim...x73265f694680>
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",'one-to-one'
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None


In [9]:
X_muestra, valores = calcular_shap(pipeline_lgbm, X_test, 2000, 42)

i = 0
prob_shap = sigmoid(valores.base_values + valores.values.sum(axis=1))    # un valor por cliente
prob_modelo = pipeline_lgbm.predict_proba(X_muestra)[:, 1]
np.allclose(prob_shap, prob_modelo, atol=1e-6)                           # debe dar True

True